# Resolve one product to one defensible URL

This is the primary supported execution path.

```text
input → interpretation → optional PCA LLM refinement → SerpAPI search
→ HTTP acquisition → local Playwright rendering → acceptance policy
→ one URL or explicit failure → evidence artifacts
```

There is **no UI, FastAPI service, Docker Compose stack, browser microservice, polling loop, `nest_asyncio`, monkey patch, or hidden compatibility layer**.

A URL is delivered only after exact identity, supplied-identifier agreement, direct-page proof, URL durability, rendered-browser access, scrapable content, and zero identity/edition conflicts.

## One-time setup

Run from the repository root:

```bash
conda env create -f environment.yml
conda activate product-url-notebook
python -m ipykernel install --user --name product-url-notebook --display-name "product-url-notebook"
python -m playwright install chromium
cp .env.example .env
jupyter lab
```

Open this notebook with the **product-url-notebook** kernel. Set `SERPAPI_API_KEY` in `.env`. Keep `PRODUCT_URL_REASONING_ENABLED=false` for the deterministic baseline.

In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the web_search_tool repository.")

ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=False)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"Repository root: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
def require_environment() -> None:
    missing = []
    if not os.getenv("SERPAPI_API_KEY", "").strip():
        missing.append("SERPAPI_API_KEY")

    reasoning_enabled = os.getenv(
        "PRODUCT_URL_REASONING_ENABLED", "false"
    ).strip().lower() in {"1", "true", "yes", "on"}

    if reasoning_enabled:
        for name in (
            "PCA_LLM_API_KEY",
            "PCA_LLM_API_VERSION",
            "PCA_LLM_ENDPOINT",
            "PCA_LLM_DEPLOYMENT",
            "PCA_LLM_CONSUMER_ID",
        ):
            if not os.getenv(name, "").strip():
                missing.append(name)

    if missing:
        raise EnvironmentError("Missing .env values: " + ", ".join(sorted(set(missing))))

    print("SERPAPI_API_KEY: configured")
    print(f"PCA reasoning enabled: {reasoning_enabled}")
    print("Browser mode: local Playwright")

require_environment()

## Product input

Only `main_text` and `country_code` are mandatory. Use `None` for unavailable optional values; never invent an EAN, retailer, or language.

In [ ]:
PRODUCT = {
    "row_id": "BOOK-1",
    "main_text": "MENSCH TÖTE DICH NICHT!",
    "country_code": "CH",
    "retailer_name": None,
    "ean": "9783311706717",
    "language_code": "de",
    "feature_set": "toy",
}

RUNTIME_OPTIONS = {
    "search_credits": 3,
    "results_per_search": 20,
    "max_candidates": 12,
    "max_per_domain": 2,
    "max_workers": 6,
    "browser_candidates": 6,
}

In [ ]:
from product_url_v2 import (
    ProductInput,
    ProductURLOrchestrator,
    evaluate_acceptance,
    load_config,
)

config = load_config(ROOT / "config" / "default.json")
product = ProductInput(
    row_id=PRODUCT["row_id"],
    main_text=PRODUCT["main_text"],
    country_code=PRODUCT["country_code"],
    retailer_name=PRODUCT["retailer_name"],
    ean=PRODUCT["ean"],
    language_code=PRODUCT["language_code"],
    feature_set=PRODUCT["feature_set"],
    runtime_options=RUNTIME_OPTIONS,
)

result = ProductURLOrchestrator(config).resolve(product)

print(f"Status: {result.decision.status.value}")
print(f"Selected URL: {result.decision.selected_url or 'None'}")
print(f"Artifacts: {result.artifact_dir}")

## Final submission row

In [ ]:
selected = next(
    (
        candidate
        for candidate in result.candidates
        if candidate.candidate_id == result.decision.selected_candidate_id
    ),
    None,
)

submission = pd.DataFrame([{
    "MAIN_TEXT": product.main_text,
    "COUNTRY": product.country_code,
    "RETAILER": product.retailer_name or "",
    "EAN": product.ean or "",
    "PRODUCT_URL": result.decision.selected_url or "",
    "DELIVERY_STATUS": result.decision.status.value,
    "CONFIDENCE": round(result.decision.confidence, 4),
    "CODING_READY": result.decision.coding_ready,
    "SOURCE_ROLE": selected.source_role.value if selected else "",
    "IDENTITY_STATUS": selected.identity_match.value if selected else "",
    "BROWSER_ACCESS": selected.browser_access.value if selected else "",
    "TEXT_EXTRACTABLE": selected.text_extractable.value if selected else "",
    "JUSTIFICATION": " ".join(result.decision.reasons),
    "ARTIFACT_DIR": result.artifact_dir,
}])
submission

## Candidate acceptance table

In [ ]:
candidate_rows = []
for candidate in result.candidates:
    verdict = evaluate_acceptance(candidate)
    candidate_rows.append({
        "selected": candidate.candidate_id == result.decision.selected_candidate_id,
        "candidate_id": candidate.candidate_id,
        "source_role": candidate.source_role.value,
        "identity": candidate.identity_match.value,
        "identifier_verified": verdict.identifier_verified,
        "direct_page": candidate.direct_product_page.value,
        "durable": candidate.durable_url.value,
        "browser": candidate.browser_access.value,
        "scrapable": candidate.text_extractable.value,
        "eligible": verdict.eligible,
        "coding": candidate.coding_evidence_complete.value,
        "confidence": round(candidate.identity_confidence, 4),
        "blockers": " | ".join(verdict.blockers),
        "url": candidate.url,
    })

candidates_df = pd.DataFrame(candidate_rows)
candidates_df

## Identity interpretation

In [ ]:
interpretation = result.interpretation
pd.DataFrame([
    {
        "field": signal.field,
        "value": signal.value,
        "confidence": signal.confidence,
        "source": signal.source,
        "exact": signal.exact,
        "evidence": signal.evidence,
    }
    for signal in (interpretation.signals if interpretation else ())
])

## Paid search actions

In [ ]:
pd.DataFrame([
    {
        "credit": observation.action.credit_number,
        "engine": observation.action.engine,
        "purpose": observation.action.purpose,
        "scope": observation.action.scope,
        "query": observation.action.query,
        "status": observation.status,
        "result_count": len(observation.results),
        "error": observation.error,
    }
    for observation in result.search_observations
])

## Observable trace

This contains reviewer-visible stages and outcomes, not hidden chain-of-thought.

In [ ]:
pd.DataFrame([
    {
        "sequence": event.sequence,
        "stage": event.stage.value,
        "event_type": event.event_type,
        "message": event.message,
    }
    for event in result.events
])

## Evidence artifacts

In [ ]:
artifact_dir = Path(result.artifact_dir)
sorted(
    str(path.relative_to(ROOT))
    for path in artifact_dir.rglob("*")
    if path.is_file()
)

In [ ]:
audit_path = artifact_dir / "audit.md"
print(audit_path.read_text(encoding="utf-8") if audit_path.is_file() else "audit.md was not created")

## Acceptance assertion

In [ ]:
delivered = result.decision.status.value in {"VERIFIED", "REVIEW_REQUIRED"}
assert delivered == bool(result.decision.selected_url)

if delivered:
    assert selected is not None
    assert evaluate_acceptance(selected).eligible

print("Notebook acceptance contract passed.")